# Phase 2: EDA与风险画像

**学习目标**
- 用 Roll Rate Analysis 从数据中验证好坏客户定义
- 掌握 Vintage 分析——风控领域最重要的资产质量评估工具
- 通过单变量分析理解每个特征与违约的关系
- 学会缺失值和异常值的业务化处理思路

**真实公司里的角色**
- 在做任何模型之前，分析师要用1-2周做"摸底分析"
- Vintage图和Roll Rate表是风控部门的"通用语言"
- 这一阶段的结论直接决定后续建模方向

**面试常问**
- "怎么从数据中验证好坏客户定义？Roll Rate怎么算？"
- "Vintage分析是什么？怎么看一张Vintage图？"
- "变量分布异常怎么处理？缺失值填什么？"

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import sys, os, warnings
warnings.filterwarnings('ignore')

sys.path.append('..')
from config import DB_PATH, BAD_DPD_THRESHOLD, GOOD_DPD_THRESHOLD

# 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')
sns.set_palette('Set2')

conn = sqlite3.connect(DB_PATH)
print('数据库已连接')

## 1. Roll Rate Analysis —— 从数据验证好坏定义

**什么是Roll Rate？**

Roll Rate = 当前处于逾期状态X的客户，在下一期"滚动"到更严重逾期状态Y的比例。

**为什么做Roll Rate？**
- 好坏客户定义应该是数据驱动的，不是拍脑袋的
- 如果一个M1客户有80%概率在6个月内滚到M3+，那M1就是"准坏客户"
- 如果大部分M2客户会还清，那M3+才是真正的坏客户

**面试必问**: "你怎么确定用M3+定义坏客户，而不是M2+或M4+？"
**答案**: 看Roll Rate。如果M2→M3的迁移率很高(>60%)，说明M2已经是事实上的坏客户了；如果很低(<30%)，那M3+才是合理的坏客户阈值。

**真实场景**: 银行每季度都要更新Roll Rate报告，监控不同逾期阶段的迁移率变化。如果某个月M1→M2的Roll Rate突然飙升，意味着下个月的M2→M3也会飙升——这是风险预警信号。

In [ ]:
# ---- Roll Rate 计算 ----
# 思路: 对每笔借据，看逾期最严重的那个月，然后看下个月的状态

repay = pd.read_sql_query('''
    SELECT loan_id, schedule_date, dpd,
           ROW_NUMBER() OVER (PARTITION BY loan_id ORDER BY schedule_date) as period
    FROM repayment_table
''', conn)

# Pivot: 行为loan_id, 列为period, 值为dpd
dpd_pivot = repay.pivot(index='loan_id', columns='period', values='dpd')
print(f'还款矩阵: {dpd_pivot.shape[0]} 笔借据 x {dpd_pivot.shape[1]} 期')

# 将DPD转为逾期阶段 (M0/M1/M2/M3/M4+)
def dpd_to_stage(dpd_val):
    if pd.isna(dpd_val): return np.nan
    if dpd_val == 0: return 'M0'
    if dpd_val <= 30: return 'M1'
    if dpd_val <= 60: return 'M2'
    if dpd_val <= 90: return 'M3'
    return 'M4+'

stage_pivot = dpd_pivot.map(dpd_to_stage)

# 计算相邻两期的Roll Rate
# 取所有相邻的月份对 (第1期->第2期, 第2期->第3期, ...)
roll_rates = []
for col in range(1, min(12, stage_pivot.shape[1])):
    flow = pd.crosstab(stage_pivot[col], stage_pivot[col+1], normalize='index')
    roll_rates.append(flow)

# 平均Roll Rate矩阵
avg_roll = sum(roll_rates) / len(roll_rates)
print('\n=== 平均 Roll Rate 矩阵 (各期的平均值) ===')
print('解读: 行=当前逾期状态, 列=下一期逾期状态')
print('例如: M1行M3列=0.15 表示 15%的M1客户下个月会滚到M3')
display(avg_roll.round(3))

In [ ]:
# ---- 可视化 Roll Rate 热力图 ----
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(avg_roll, annot=True, fmt='.1%', cmap='YlOrRd', 
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('Roll Rate Matrix: 平均月度逾期迁移率', fontsize=14, fontweight='bold')
ax.set_xlabel('下一期逾期状态')
ax.set_ylabel('当前逾期状态')
plt.tight_layout()
plt.show()

# 关键解读
print('=== Roll Rate 关键发现 ===')
print('1. M0保持在M0的比例: {:.1%} (越高越好)'.format(avg_roll.loc['M0','M0'] if 'M0' in avg_roll.index else 0))
print('2. M2滚到M3+的比例: 看M2行M3+M4+列的和')
print('3. M3回到M0的可能性: 几乎为0 (一旦M3+, 基本不可能还清)')
print()
print('结论: 如果M2->M3的迁移率很高, 你可以考虑用M2+定义坏客户;')
print('      如果M2->M3的迁移率低, M3+才是有意义的坏客户阈值')

## 2. Vintage 分析 —— 风控最重要的分析工具

**什么是Vintage分析？**

按放款月份分组，看每个"账龄"(Month-on-Book, MOB)的累计坏账率变化。

```
放款月份=2018-01:  MOB1: 0.5% -> MOB3: 2.1% -> MOB6: 3.8% -> MOB12: 5.2%
放款月份=2018-06:  MOB1: 0.8% -> MOB3: 3.5% -> MOB6: 6.2% -> MOB12: 8.1%  <- 资产恶化!
放款月份=2019-01:  MOB1: 0.3% -> MOB3: 1.5% -> MOB6: 2.8% -> MOB12: ...
```

**怎么看Vintage图？**
1. 曲线的陡峭程度 = 坏账暴露速度（越陡 = 客户质量越差）
2. 曲线的最终高度 = 该批次最终的坏账率
3. 不同月份曲线之间的差距 = 资产质量的变化趋势

**面试怎么问**: "给你一张Vintage图，怎么看？"
**标准答案**: 先看趋势——曲线是收敛还是发散？收敛说明风控策略稳定，发散说明资产质量在恶化。再看MOB6/MOB12的坏账率——这是业界对标的标准指标。最后看最近几个月——如果最近几个月的Vintage曲线明显高于历史，需要立刻排查原因。

In [ ]:
# ---- Vintage 分析 ----
# 思路: 对每笔借据,按放款月份分组,计算每个MOB的累计坏账率

loans = pd.read_sql_query('''
    SELECT l.loan_id, l.issue_date, l.loan_status
    FROM loan_table l
''', conn)
loans['issue_date'] = pd.to_datetime(loans['issue_date'])
loans['issue_month'] = loans['issue_date'].dt.to_period('M')
loans['is_bad'] = (loans['loan_status'] == 'Bad').astype(int)

# 需要还款数据来计算每个MOB的累计坏账
repay_detail = pd.read_sql_query('''
    SELECT r.loan_id, r.schedule_date, r.dpd,
           l.issue_date, l.loan_status
    FROM repayment_table r
    JOIN loan_table l ON r.loan_id = l.loan_id
''', conn)
repay_detail['issue_date'] = pd.to_datetime(repay_detail['issue_date'])
repay_detail['schedule_date'] = pd.to_datetime(repay_detail['schedule_date'])
repay_detail['mob'] = (
    (repay_detail['schedule_date'].dt.year - repay_detail['issue_date'].dt.year) * 12 +
    (repay_detail['schedule_date'].dt.month - repay_detail['issue_date'].dt.month)
)
repay_detail['issue_month'] = repay_detail['issue_date'].dt.to_period('M')

# 对每个放款月份+MOB组合, 计算累计坏账率
vintage_data = []
for (issue_m, mob), grp in repay_detail.groupby(['issue_month', 'mob']):
    if mob < 1 or mob > 12:
        continue
    total_loans = grp['loan_id'].nunique()
    # 累计: MOB内有过90+逾期的就算坏
    bad_loans = grp[grp['dpd'] >= 90]['loan_id'].nunique()
    bad_rate = bad_loans / total_loans if total_loans > 0 else 0
    vintage_data.append({
        'issue_month': str(issue_m),
        'mob': mob,
        'bad_rate': bad_rate,
        'total_loans': total_loans
    })

vintage_df = pd.DataFrame(vintage_data)
print(f'Vintage数据: {len(vintage_df)} 条 (issue_month x MOB)')

# 只保留最近12个放款月份
recent_months = sorted(vintage_df['issue_month'].unique())[-8:]
vintage_plot = vintage_df[vintage_df['issue_month'].isin(recent_months)]

In [ ]:
# ---- Vintage 曲线图 ----
fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(recent_months)))
for i, month in enumerate(recent_months):
    data = vintage_plot[vintage_plot['issue_month'] == month].sort_values('mob')
    ax.plot(data['mob'], data['bad_rate'] * 100, 'o-', color=colors[i],
            linewidth=1.8, markersize=4, label=month)

ax.set_xlabel('MOB (Month-on-Book, 账龄)', fontsize=12)
ax.set_ylabel('Cumulative Bad Rate (%)', fontsize=12)
ax.set_title('Vintage Analysis: 各放款月份累计坏账率曲线', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.5, 12.5)
plt.tight_layout()
plt.show()

print('=== Vintage 图解读指南 ===')
print('1. 曲线越陡 = 坏账暴露越快, 客群质量越差')
print('2. 如果后面的月份曲线整体低于前面的 -> 风控在改善')
print('3. 如果后面的月份曲线整体高于前面的 -> 风控在放松, 资产恶化')
print('4. MOB6是一个重要的benchmark点: 大部分坏账在6个月内已经暴露')
print('5. 银行通常用 MOB12 Vintage Bad Rate 作为最终损失率的估计')

## 3. 单变量分析 —— 每个变量和违约的关系

在做模型之前，需要逐个理解每个变量和坏账之间的关系。这一步帮你：
1. 发现哪些变量有区分度（初步判断IV会不会高）
2. 识别变量和违约的关系是否合理（单调递增/递减？U型？）
3. 为后续分箱提供方向

**面试中常问**: "给你一个变量，你怎么判断它有没有预测力？"
**标准回答**: 先看分布——好人和坏人在这个变量上的分布有没有明显错开？再看单调性——坏账率是否随着变量值单调变化？如果都满足，这个变量大概率是个好特征。

In [ ]:
# ---- 加载完整数据用于分析 ----
df = pd.read_sql_query('''
    SELECT 
        l.loan_id, l.loan_status, l.loan_amnt, l.term, l.int_rate, l.fico_score,
        a.annual_inc, a.dti, a.credit_age, a.revol_util, 
        a.inq_6m, a.delinq_2yrs, a.pub_rec, a.purpose, a.home_ownership, a.emp_length
    FROM loan_table l
    JOIN apply_table a ON l.application_id = a.application_id
''', conn)

df['is_bad'] = (df['loan_status'] == 'Bad').astype(int)
print(f'分析样本: {len(df):,} 笔借据, 坏账率: {df["is_bad"].mean():.2%}')
print(f'变量列表: {list(df.columns)}')

In [ ]:
# ---- 数值型变量: 分箱后看坏账率趋势 ----
num_vars = ['fico_score', 'annual_inc', 'dti', 'credit_age', 
            'revol_util', 'inq_6m', 'delinq_2yrs', 'loan_amnt', 'int_rate']

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()

for i, var in enumerate(num_vars):
    ax = axes[i]
    data = df[[var, 'is_bad']].dropna()
    
    # 等频分10箱
    data['bin'] = pd.qcut(data[var], q=10, duplicates='drop')
    
    # 每箱统计
    bin_stats = data.groupby('bin').agg(
        bad_rate=('is_bad', 'mean'),
        cnt=('is_bad', 'count')
    ).reset_index()
    bin_stats['bin_label'] = bin_stats['bin'].astype(str)
    
    # 柱状图(样本量) + 折线图(坏账率)
    ax2 = ax.twinx()
    bars = ax.bar(range(len(bin_stats)), bin_stats['cnt'], alpha=0.3, color='steelblue')
    line = ax2.plot(range(len(bin_stats)), bin_stats['bad_rate'] * 100, 
                    'ro-', linewidth=2, markersize=6)
    
    ax.set_xlabel(var, fontsize=10)
    ax.set_ylabel('Count', color='steelblue')
    ax2.set_ylabel('Bad Rate (%)', color='red')
    ax.set_title(f'{var} vs Bad Rate', fontsize=11, fontweight='bold')
    ax.set_xticks(range(len(bin_stats)))
    ax.set_xticklabels(range(1, len(bin_stats)+1), rotation=0, fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ---- 类别型变量: 看各类别的坏账率 ----
cat_vars = ['purpose', 'home_ownership', 'emp_length', 'term']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, var in enumerate(cat_vars):
    ax = axes[i]
    stats = df.groupby(var).agg(
        bad_rate=('is_bad', 'mean'),
        cnt=('is_bad', 'count')
    ).sort_values('bad_rate', ascending=False)
    
    colors = ['#d73027' if r > df['is_bad'].mean() else '#4575b4' 
              for r in stats['bad_rate']]
    bars = ax.barh(range(len(stats)), stats['bad_rate'] * 100, color=colors, alpha=0.8)
    
    for j, (rate, cnt) in enumerate(zip(stats['bad_rate'], stats['cnt'])):
        ax.text(rate * 100 + 0.3, j, f'{rate*100:.1f}% (n={cnt:,})', va='center', fontsize=8)
    
    ax.set_yticks(range(len(stats)))
    ax.set_yticklabels(stats.index, fontsize=9)
    ax.set_xlabel('Bad Rate (%)')
    ax.set_title(f'{var} vs Bad Rate (dashed=overall avg)', fontsize=11, fontweight='bold')
    ax.axvline(df['is_bad'].mean() * 100, color='gray', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

## 4. 缺失值分析

风控数据几乎不可能没有缺失值。关键不是"有没有缺失"，而是"缺失本身有没有信息量"。

**面试常问**: "缺失值怎么处理？"
**进阶答案**: 不只填均值/中位数。先分析缺失是否有业务含义——比如"收入缺失"可能意味着自由职业者或不愿意透露收入的人，这类人本身风险就不同。如果可以验证，把"缺失"作为一个单独的类别放进模型，比填均值效果好得多。

In [ ]:
# ---- 缺失值分析 ----
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100)
missing_df = pd.DataFrame({
    'variable': missing.index,
    'missing_count': missing.values,
    'missing_pct': missing_pct.values
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

if len(missing_df) > 0:
    display(missing_df)
    print(f'\n共 {len(missing_df)} 个变量有缺失值')
else:
    print('当前合成数据无缺失值。但在真实数据中，以下变量通常有缺失：')
    print('  - annual_inc: 约5-10% (自报收入，部分用户不填)')
    print('  - dti: 约3-5% (计算DTI需要收入数据)')
    print('  - revol_util: 约1-2% (无循环信用的用户此项为空)')
    print('  - emp_length: 约5-8% (失业/自由职业者不填)')

print('\n=== 缺失值处理策略 (按业务含义分类) ===')
print('类型1: 结构性缺失 (如: 没有循环信用 -> revol_util为空)')
print('  -> 填充0或单独类别 "NoRevolving"')
print('类型2: 随机缺失 (如: 系统录入遗漏)')
print('  -> 中位数/众数填充, 或建模预测填充')
print('类型3: 信息性缺失 (如: 故意不填收入, 可能是高风险)')
print('  -> 保留缺失标记作为特征, 同时填充特殊值(-9999)')

## 5. 异常值检测与处理

异常值不等于错误值。风控里的"异常"有时候恰恰是最有信息量的。

**面试关键区分**:
- 数据错误(Data Error): 年龄=200, 收入=-10000 → 必须修正或剔除
- 业务极端值(Business Extreme): FICO=850, 收入=500万 → 合法但极端, 需要缩尾(winsorize)
- 风险信号(Risk Signal): 查询次数=10次 → 看起来异常, 但恰恰是高风险信号, 不能删！

In [ ]:
# ---- 异常值检测 (IQR方法 + 业务规则) ----
print('=== 数值型变量分布统计 ===')
stats_df = df[num_vars].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
stats_df['IQR'] = stats_df['75%'] - stats_df['25%']
stats_df['lower_fence'] = stats_df['25%'] - 1.5 * stats_df['IQR']
stats_df['upper_fence'] = stats_df['75%'] + 1.5 * stats_df['IQR']
stats_df['outlier_pct'] = 0.0

for var in num_vars:
    lower = stats_df.loc[var, 'lower_fence']
    upper = stats_df.loc[var, 'upper_fence']
    outlier_ratio = ((df[var] < lower) | (df[var] > upper)).mean()
    stats_df.loc[var, 'outlier_pct'] = outlier_ratio

display(stats_df[['mean', 'std', '1%', '99%', 'IQR', 'outlier_pct']].round(2))

print('\n=== 处理建议 ===')
print('1. outlier_pct < 1%: 可以直接做 1%/99% winsorize(缩尾)')
print('2. outlier_pct 1%-5%: 检查异常值的业务含义, 再决定')
print('3. outlier_pct > 5%: 不是异常值, 是长尾分布, 做对数变换或分箱')
print('4. 特别注意: inq_6m/delinq_2yrs 的高值可能是风险信号, 不要盲目缩尾!')

In [ ]:
# ---- 箱线图: 分好坏客户看分布差异 ----
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

plot_vars = ['fico_score', 'annual_inc', 'dti', 'credit_age', 
             'revol_util', 'inq_6m', 'delinq_2yrs', 'int_rate']

for i, var in enumerate(plot_vars):
    ax = axes[i]
    good_data = df[df['is_bad'] == 0][var].dropna()
    bad_data = df[df['is_bad'] == 1][var].dropna()
    
    bp = ax.boxplot([good_data, bad_data], labels=['Good', 'Bad'], 
                    patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('#4575b4')
    bp['boxes'][1].set_facecolor('#d73027')
    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.set_ylabel('Value')

plt.suptitle('Good vs Bad: Variable Distribution Comparison', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('=== 读图要点 ===')
print('看两个箱体的重叠程度:')
print('  - 重叠越少 -> 该变量的区分度越好 (如fico_score)')
print('  - 几乎完全重叠 -> 该变量可能没有预测力')
print('  - 中位数差距大 -> 好特征!')

In [ ]:
# ---- 相关性矩阵 ----
corr_vars = ['fico_score', 'annual_inc', 'dti', 'credit_age', 
             'revol_util', 'inq_6m', 'delinq_2yrs', 'loan_amnt', 
             'int_rate', 'is_bad']
corr = df[corr_vars].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Variable Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 与目标变量的相关性排序
target_corr = corr['is_bad'].drop('is_bad').sort_values(ascending=False)
print('=== 各变量与 Bad 的相关性 (Pearson) ===')
for var, val in target_corr.items():
    direction = '正相关(危险)' if val > 0 else '负相关(安全)'
    print(f'  {var:20s}: {val:+.3f}  ({direction})')

## Phase 2 总结

### 你学会了什么
1. **Roll Rate Analysis**: 用数据驱动的方式验证好坏客户定义，理解M0-M6的迁移逻辑
2. **Vintage Analysis**: 风控最重要的资产质量评估工具——按放款月份追踪累计坏账率
3. **单变量分析**: 逐个理解变量与违约的关系，判断区分度和单调性
4. **缺失值处理**: 不只填均值——理解"缺失"本身是否有信息量
5. **异常值处理**: 区分数据错误 vs 业务极端值 vs 风险信号

### 简历可写（追加到 Phase 1 后面）
> 完成信贷资产深度风险画像，通过Roll Rate Analysis验证M3+坏客户定义，
> 使用Vintage分析追踪8个放款批次的累计坏账率变化趋势，
> 对9个数值变量和4个类别变量完成单变量风险分析，
> 识别出FICO分/近6月查询次数/循环利率利用率等高区分度特征。

### 面试中怎么讲这一段
> "在建模前我先做了完整的EDA。用Roll Rate验证了M3+的坏客户定义——M2到M3+的迁移率超过XX%，说明M3+是合理的阈值。然后做了Vintage分析，发现各月曲线基本收敛，风控策略比较稳定。最后通过单变量分析筛出了FICO分、DTI等几个高区分度的变量......"

### 下一阶段
Phase 3: 特征工程与WOE/IV —— 把原始变量转化为评分卡可用的风险特征

In [ ]:
conn.close()
print('数据库连接已关闭')